In [ ]:
from transformers import pipeline

# Main model for the whole notebook
generator = pipeline("text-generation", model="gpt2")

def ask(question):
    result = generator(question, max_length=150, num_return_sequences=1)
    return result[0]['generated_text']

# Test it
print(ask("Artificial intelligence is"))

In [13]:
from transformers import AutoTokenizer

text = "Generative AI is transforming education."

# BERT tokenizer
tokenizer1 = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_tokens = tokenizer1.tokenize(text)
print("BERT tokens:", bert_tokens)
print("BERT count:", len(bert_tokens))

# GPT2 tokenizer
tokenizer2 = AutoTokenizer.from_pretrained("gpt2")
gpt2_tokens = tokenizer2.tokenize(text)
print("\nGPT2 tokens:", gpt2_tokens)
print("GPT2 count:", len(gpt2_tokens))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT tokens: ['genera', '##tive', 'ai', 'is', 'transforming', 'education', '.']
BERT count: 7

GPT2 tokens: ['Gener', 'ative', 'ĠAI', 'Ġis', 'Ġtransforming', 'Ġeducation', '.']
GPT2 count: 7


#Day 2 Reflection
###BERT tokens: ['genera', '##tive', 'ai', 'is', 'transforming', 'education', '.']
###GPT2 tokens: ['Gener', 'ative', 'ĠAI', 'Ġis', 'Ġtransforming', 'Ġeducation', '.']

Both tokenizers gave 7 tokens for this sentence.
BERT uses WordPiece — splits "Generative" into "genera" + "##tive", lowercases everything.
GPT2 uses BPE — keeps capital letters, adds "Ġ" symbol before words that follow a space.
For longer or uncommon words, GPT2 usually produces more tokens than BERT

In [14]:
question = "If a car travels 60 km in 1 hour, how far in 5 hours?"

# Normal prompt
response1 = generator(question, max_new_tokens=50)
print("Normal:", response1[0]['generated_text'])

print("---")

# Chain-of-thought prompt
response2 = generator(question + " Think step by step.", max_new_tokens=80)
print("CoT:", response2[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Normal: If a car travels 60 km in 1 hour, how far in 5 hours?

Well, the following formula is taken from the current law:

In a given month, in a given calendar year, a vehicle traveling 60 kilometres in 1 hour, or 4.5 miles per hour, would travel between 1,000
---
CoT: If a car travels 60 km in 1 hour, how far in 5 hours? Think step by step.

The main reason is the speed. The car driver keeps the speed constant with every motorized vehicle on the road. The speed is not fixed.

The car driver's speed is the speed that you are driving. The speed is the speed that you are traveling.

In the US, the speed of a car is calculated as the average speed of the vehicle, with the maximum


#Normal prompt:
 GPT2 tried to answer but went off-track,
               talking about laws and formulas instead of
               giving a direct answer.

#CoT prompt:
Also didn't calculate correctly, but stayed
            more focused on the concept of speed.

##Note: GPT2 is a small old model,it struggles with math.
A larger model like Gemini would give:
"60 km/hr × 5 hrs = 300 km" clearly.

#Lesson:
Chain-of-thought prompting helps guide the model to think more logically, even if the model is limited.

In [16]:
# Summarization using GPT2 (text-generation)
text = """
Artificial intelligence is transforming every industry in the world.
From healthcare to education, AI systems are helping humans make better
decisions faster. Machine learning models can now detect diseases from
medical scans with accuracy matching expert doctors. In education, AI
tutors provide personalized learning experiences for every student.
However, AI also raises concerns about job displacement and privacy.
Experts say the key is to use AI as a tool that assists humans,
not replaces them.
"""

prompt = "Summarize this text in 2 sentences: " + text

result = generator(prompt, max_new_tokens=80)
print("Summary:", result[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary: Summarize this text in 2 sentences: 
Artificial intelligence is transforming every industry in the world. 
From healthcare to education, AI systems are helping humans make better 
decisions faster. Machine learning models can now detect diseases from 
medical scans with accuracy matching expert doctors. In education, AI 
tutors provide personalized learning experiences for every student. 
However, AI also raises concerns about job displacement and privacy. 
Experts say the key is to use AI as a tool that assists humans, 
not replaces them.
A recent study by  Gadget and a team from the University of Oxford found that  a big portion of  the workforce is already employed
in fields such as medicine, engineering, and science.
AI technologies are changing everything in the world.  In fact, one major study  of the 
research that  has been done on AI  shows


#Day 3 Summarizaiton Reflection
##Basic prompt:
 GPT2 continued the text rather than summarizing it, because it is a text generator, not a summarizer.

##Improved prompt result:
 Slightly better but still drifted.

##Lesson:
 GPT2 is not designed for summarization.
A proper summarization model (like BART) or Gemini would give a clean 2-sentence summary.
Prompt quality matters adding "in 2 sentences" helped focus the output.

In [17]:
# Simple chatbot using GPT2
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("gpt2")
chat_model = AutoModelForCausalLM.from_pretrained("gpt2")

conversation_history = ""

print("Chatbot ready! Type 'quit' to stop.\n")

for _ in range(3):  # 3 turns only (no infinite loop in Colab)
    user = input("You: ")
    if user.lower() == "quit":
        break

    conversation_history += f"User: {user}\nBot:"

    inputs = tokenizer.encode(conversation_history, return_tensors="pt")
    outputs = chat_model.generate(inputs, max_new_tokens=50, pad_token_id=50256)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    bot_reply = response[len(conversation_history):]
    print("Bot:", bot_reply)
    conversation_history += bot_reply + "\n"

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot ready! Type 'quit' to stop.

You: Hello, How are you doing ?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Bot:  I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I
You: Good to hear this from you.
Bot:  I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I'm fine.
Bot: I


KeyboardInterrupt: Interrupted by user

#Day 3 Chatbot Reflection
Chatbot worked and maintained conversation history.GPT2 repeated "I'm fine" many times this is called "repetition problem" in small language models.

Larger models like Gemini or GPT-4 handle conversation much better by tracking context properly.

###Lesson:
 Conversation history management is important for chatbots. Model size directly affects response quality.

In [18]:
# Step 1: Documents
docs = [
    "AI stands for Artificial Intelligence.",
    "Machine learning is a subset of AI.",
    "GANs generate synthetic data.",
    "Deep learning uses neural networks with many layers.",
    "Natural Language Processing helps computers understand text.",
    "Computer vision allows machines to interpret images.",
    "Transformers are the architecture behind modern LLMs.",
    "Reinforcement learning trains agents through rewards.",
    "Embeddings represent words as numerical vectors.",
    "RAG combines retrieval with generation for better answers."
]

# Step 2: Create Embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(docs)

print("Embeddings created!")
print("Shape:", doc_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created!
Shape: (10, 384)


In [19]:
# Step 3: Search relevant document using query
query = "What is AI?"
query_embedding = embed_model.encode([query])

scores = np.dot(doc_embeddings, query_embedding.T)
best_index = np.argmax(scores)

print("Query:", query)
print("Best matching document:", docs[best_index])
print("Score:", scores[best_index][0])

print("\n--- Try another query ---")
query2 = "How do neural networks work?"
query_embedding2 = embed_model.encode([query2])
scores2 = np.dot(doc_embeddings, query_embedding2.T)
best_index2 = np.argmax(scores2)
print("Query:", query2)
print("Best matching document:", docs[best_index2])

Query: What is AI?
Best matching document: AI stands for Artificial Intelligence.
Score: 0.8755641

--- Try another query ---
Query: How do neural networks work?
Best matching document: Deep learning uses neural networks with many layers.


In [20]:
# Full RAG Pipeline — Retrieve + Generate Answer
def rag_answer(question):
    # Step 1: Find best document
    q_embedding = embed_model.encode([question])
    scores = np.dot(doc_embeddings, q_embedding.T)
    best_doc = docs[np.argmax(scores)]

    # Step 2: Generate answer using context
    prompt = f"Context: {best_doc}\nQuestion: {question}\nAnswer:"
    result = generator(prompt, max_new_tokens=60)
    answer = result[0]['generated_text']

    print("Question:", question)
    print("Retrieved doc:", best_doc)
    print("Answer:", answer)
    print("-" * 50)

# Test with 5 different questions
rag_answer("What is AI?")
rag_answer("What are embeddings?")
rag_answer("How does RAG work?")
rag_answer("What is deep learning?")
rag_answer("What is computer vision?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is AI?
Retrieved doc: AI stands for Artificial Intelligence.
Answer: Context: AI stands for Artificial Intelligence.
Question: What is AI?
Answer: Artificial intelligence is a science. In the United States, in this field we have the most advanced artificial intelligence technology that is now available. There are some scientists who think that we are the next great civilization. And the good news is, that's probably true. The good news is, we are getting
--------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are embeddings?
Retrieved doc: Embeddings represent words as numerical vectors.
Answer: Context: Embeddings represent words as numerical vectors.
Question: What are embeddings?
Answer: The term embeddings is used to distinguish an embeddable representation of data. In fact, the notion of embedding is a generalization of embedding to a number of other ways of describing data.
In the simplest case, you would have data like this:
{ :name =>
--------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does RAG work?
Retrieved doc: RAG combines retrieval with generation for better answers.
Answer: Context: RAG combines retrieval with generation for better answers.
Question: How does RAG work?
Answer: RAG supports RAG's new type system in a number of ways, and this includes two key goals:
We create a new type system that provides a new way to understand which questions are questions that can be answered using the same type classes. We also introduce a new type class that provides a
--------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is deep learning?
Retrieved doc: Deep learning uses neural networks with many layers.
Answer: Context: Deep learning uses neural networks with many layers.
Question: What is deep learning?
Answer: Deep neural networks are general computing systems that are able to predict and process data. For example, a neural network can predict the number of hits on a website. The idea is to learn how to calculate how many hits a website has to deliver a certain amount of traffic. Once a certain amount of traffic
--------------------------------------------------
Question: What is computer vision?
Retrieved doc: Computer vision allows machines to interpret images.
Answer: Context: Computer vision allows machines to interpret images.
Question: What is computer vision?
Answer: Computer vision allows machines to interpret images. Question: How do computers work?
Answer: Computers work by using a computer's information processor (CPU).
Question: What is computer vision?
Answer: Computer v

#Day 5 Reflection
RAG pipeline worked successfully.
- "What is AI?" → correctly retrieved "AI stands for Artificial Intelligence."
- "What are embeddings?" → correctly retrieved embeddings document
- All 5 questions retrieved the correct document

GPT2 answers drifted off-topic after retrieving context
because it is a text generator, not a question-answerer.
A better model (Gemini/GPT-4) would give clean answers.

###Lesson:
 The RETRIEVAL part of RAG worked perfectly.
The generation quality depends on the model used.

In [21]:
import torch
import torch.nn as nn

# Generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Tanh()
        )
    def forward(self, x):
        return self.model(x)

print("Generator defined!")

Generator defined!


In [22]:
# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

print("Discriminator defined!")

Discriminator defined!


In [23]:
# Generate a fake image
noise = torch.randn(1, 100)
gen = Generator()
fake_image = gen(noise)

print("Fake image shape:", fake_image.shape)
print("Min value:", fake_image.min().item())
print("Max value:", fake_image.max().item())
print("GAN working! Generator created a fake 28x28 image.")

Fake image shape: torch.Size([1, 784])
Min value: -0.5871524214744568
Max value: 0.6423730850219727
GAN working! Generator created a fake 28x28 image.


#Day 6 Reflection
Generator created a fake image of shape (1, 784)
which represents a 28x28 pixel image (like MNIST digits).

###Generator role:
 Takes random noise (100 numbers)
and transforms it into a fake image (784 numbers).

###Discriminator role:
 Takes an image (784 numbers)
and outputs 1 number between 0-1.
(1 = real image, 0 = fake image)

###They compete:
 Generator tries to fool Discriminator,
Discriminator tries to catch fakes.
Over time, Generator produces increasingly realistic images.

###GAN vs Diffusion:
GANs are faster but harder to train.
Diffusion models (like Stable Diffusion) are slower
but produce higher quality images today.

In [24]:
from transformers import pipeline

# Tool 1: Token Counter
def count_tokens(text):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("gpt2")
    tokens = tok.tokenize(text)
    return len(tokens), tokens

# Tool 2: Summarizer
def summarize(text):
    prompt = "Summarize in 2 sentences: " + text
    result = generator(prompt, max_new_tokens=60)
    return result[0]['generated_text']

# Tool 3: Sentiment Analysis
classifier = pipeline("sentiment-analysis")

# ---- Run the combined tool ----
sample_text = "Artificial intelligence is transforming industries worldwide. Many companies are investing heavily in AI research and development."

print("=" * 50)
print("GENAI TEXT ANALYSIS TOOL")
print("=" * 50)

count, tokens = count_tokens(sample_text)
print(f"\n1. TOKEN COUNT: {count}")
print(f"   Tokens: {tokens}")

print(f"\n2. SUMMARY:")
print(summarize(sample_text))

print(f"\n3. SENTIMENT:")
sentiment = classifier(sample_text)
print(f"   Label: {sentiment[0]['label']}")
print(f"   Score: {round(sentiment[0]['score'] * 100, 2)}%")

print("\n" + "=" * 50)
print("Analysis Complete!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

GENAI TEXT ANALYSIS TOOL


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



1. TOKEN COUNT: 19
   Tokens: ['Art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġtransforming', 'Ġindustries', 'Ġworldwide', '.', 'ĠMany', 'Ġcompanies', 'Ġare', 'Ġinvesting', 'Ġheavily', 'Ġin', 'ĠAI', 'Ġresearch', 'Ġand', 'Ġdevelopment', '.']

2. SUMMARY:
Summarize in 2 sentences: Artificial intelligence is transforming industries worldwide. Many companies are investing heavily in AI research and development. At the same time, Artificial Intelligence has been successfully used by a wide range of industries. At the same time, Artificial Intelligence has been successfully used by a wide range of industries. As of December 2015, over 3.5 billion jobs have been created worldwide. An estimated 5.7 billion people (

3. SENTIMENT:
   Label: POSITIVE
   Score: 99.93%

Analysis Complete!


#Final Reflection
This week I learned how LLMs process text through
tokenization, how prompts affect model output, and
how RAG retrieves relevant documents to improve answers.
I built a chatbot, a RAG pipeline with 10 documents,
a GAN that generates fake images, and a final text
analysis tool combining token counting, summarization,
and sentiment analysis. The hardest part was fixing
API quota issues with Gemini. Hugging Face models
worked best as a free alternative. GPT2 struggled
with math and summarization — bigger models perform
significantly better on complex tasks.